# InvokeAI on Google Colab — persistent setup (Google Drive)

Runs the latest InvokeAI release (web UI) on a Colab GPU, with all state stored on
Google Drive so nothing is lost when the runtime stops and you start it again.

Just **two steps**: run **Start** to launch, run **Stop** when done.

**What persists on Drive** (`INVOKEAI_ROOT`, default `My Drive/invokeai`): downloaded
models, generated images (`outputs`), the database (boards, gallery, workflows) and
`invokeai.yaml`. The InvokeAI package itself is reinstalled into Colab's fast local disk
each fresh runtime (a few minutes) — only the data lives on Drive.

**Drive location**: the Start cell has form fields `DRIVE_NAME` (default `My Drive`; or a
Shared Drive name) and `INVOKEAI_FOLDER` (default `invokeai`) so you can choose where the
data goes.

**Password**: set `AUTH_USER` / `AUTH_PASSWORD` in the Start form to put HTTP Basic Auth
(via a Caddy reverse proxy) in front of the public link. Leave `AUTH_PASSWORD` empty for
an unprotected link.

**Downloading models**: do it inside the running app — **Model Manager** lets you paste a
HuggingFace / CivitAI / direct URL and InvokeAI downloads it straight into the Drive root
itself. For CivitAI, add your API key in the Model Manager settings first. No extra cell
needed.

## GPU
Use the most powerful GPU Google offers: **A100 (40 GB)** — the only Colab GPU that fits
the large modern models (Flux.1/Flux.2, SD 3.5 Large, Qwen Image) without heavy
offloading. Requires Colab Pro / Pro+. Fallback order: A100 > L4 (24 GB) > T4. Set it in
`Runtime -> Change runtime type -> A100 GPU` before running Start.

## How to use
1. Select an A100 (or L4) GPU runtime.
2. Run **Step 1 — Start** (authorize Google Drive when asked). It installs InvokeAI,
   launches it in the background and prints a public `trycloudflare.com` URL — open it
   once the server is up.
3. When finished, run **Step 2 — Stop**. To resume later, just run Start again; your
   models and images are still on Drive.

## Step 1 — Start

Mounts Drive, installs the latest InvokeAI, and launches the web UI **in the background**.
Pick the Drive location with `DRIVE_NAME` / `INVOKEAI_FOLDER` and optionally set
`AUTH_USER` / `AUTH_PASSWORD` to password-protect the link. Authorize Google Drive when
prompted, then open the printed `trycloudflare.com` URL once it reports the server is up.

In [ ]:
# Step 1 - Start: check env, mount Drive, install latest InvokeAI, launch the web UI.
# The server and tunnel run in the BACKGROUND, so the cell finishes and you can run
# Step 2 - Stop later. Rerun this cell any time to start a fresh session.

# Where on Google Drive to keep InvokeAI data (edit in the form on the right).
DRIVE_NAME = "My Drive"  # @param {type:"string"}
INVOKEAI_FOLDER = "invokeai"  # @param {type:"string"}
# Password-protect the public link (HTTP Basic Auth). Leave AUTH_PASSWORD empty for no auth.
AUTH_USER = "invoke"  # @param {type:"string"}
AUTH_PASSWORD = "invoke"  # @param {type:"string"}

import os
import sys
import re
import time
import subprocess
import urllib.request
import urllib.error

# --- Environment check ---
print(f"Python: {sys.version.split()[0]}")
if not (3, 11) <= sys.version_info[:2] <= (3, 12):
    print("WARNING: InvokeAI needs Python 3.11-3.12; this runtime may fail to install.")
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip()
    print(f"GPU: {gpu}")
    if "A100" not in gpu:
        print("WARNING: A100 not detected. For large models pick 'A100 GPU' in "
              "Runtime -> Change runtime type (Colab Pro/Pro+).")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("ERROR: No NVIDIA GPU. Set Runtime -> Change runtime type -> A100 GPU, then rerun.")

# --- Mount Google Drive and pin INVOKEAI_ROOT there for persistence ---
from google.colab import drive
drive.mount("/content/drive")

# "My Drive" is your personal drive; any other value is treated as a Shared Drive name.
drive_name = DRIVE_NAME.strip()
if drive_name.lower().replace(" ", "") in ("mydrive", ""):
    drive_base = "/content/drive/MyDrive"
else:
    drive_base = f"/content/drive/Shareddrives/{drive_name}"
if not os.path.isdir(drive_base):
    raise FileNotFoundError(
        f"Drive path not found: {drive_base}. Set DRIVE_NAME to 'My Drive' for your "
        f"personal drive, or to the exact name of a Shared Drive you can access."
    )

os.environ["INVOKEAI_ROOT"] = os.path.join(drive_base, INVOKEAI_FOLDER)
os.environ["INVOKEAI_HOST"] = "127.0.0.1"
os.environ["INVOKEAI_PORT"] = "9090"
os.makedirs(os.environ["INVOKEAI_ROOT"], exist_ok=True)
print(f"INVOKEAI_ROOT = {os.environ['INVOKEAI_ROOT']}")

# --- Install the latest InvokeAI (skip if already present this session) ---
try:
    import invokeai.version
    print(f"InvokeAI already installed: {invokeai.version.__version__}")
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", "invokeai[cuda]",
         "--extra-index-url", "https://download.pytorch.org/whl/cu128"],
        check=True,
    )

# --- Fix Colab's protobuf: diffusers needs >=5.26 ('runtime_version'); Colab ships older ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf>=5.26,<6"], check=True)

# --- Download cloudflared and caddy (idempotent) ---
if not os.path.exists("/usr/local/bin/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared",
    )
    os.chmod("/usr/local/bin/cloudflared", 0o755)
if AUTH_PASSWORD and not os.path.exists("/usr/local/bin/caddy"):
    urllib.request.urlretrieve(
        "https://caddyserver.com/api/download?os=linux&arch=amd64", "/usr/local/bin/caddy"
    )
    os.chmod("/usr/local/bin/caddy", 0o755)

port = os.environ["INVOKEAI_PORT"]
proxy_port = "9091"
server_log = "/content/invokeai.log"

# --- Launch InvokeAI in the background ---
subprocess.Popen(["invokeai-web"], stdout=open(server_log, "w"), stderr=subprocess.STDOUT)

# Wait until the server is listening (a response, even 404, means it is up).
print("Starting InvokeAI server (first launch can take a few minutes)...")
server_ready = False
for _ in range(120):
    time.sleep(5)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=5)
    except urllib.error.HTTPError:
        server_ready = True
        break
    except Exception:
        continue
    server_ready = True
    break

if not server_ready:
    print(f"ERROR: server did not start. Last lines of {server_log}:")
    print(subprocess.run(["tail", "-n", "30", server_log], capture_output=True, text=True).stdout)
else:
    print("InvokeAI server is up.")
    # Optional HTTP Basic Auth in front of InvokeAI via a Caddy reverse proxy.
    if AUTH_PASSWORD:
        pw_hash = subprocess.run(
            ["/usr/local/bin/caddy", "hash-password", "--plaintext", AUTH_PASSWORD],
            capture_output=True, text=True,
        ).stdout.strip()
        caddyfile = (
            f":{proxy_port} {{\n"
            f"\tbasic_auth {{\n"
            f"\t\t{AUTH_USER} {pw_hash}\n"
            f"\t}}\n"
            f"\treverse_proxy 127.0.0.1:{port}\n"
            f"}}\n"
        )
        with open("/content/Caddyfile", "w") as caddy_conf:
            caddy_conf.write(caddyfile)
        subprocess.Popen(
            ["/usr/local/bin/caddy", "run", "--config", "/content/Caddyfile", "--adapter", "caddyfile"],
            stdout=open("/content/caddy.log", "w"), stderr=subprocess.STDOUT,
        )
        # Wait for Caddy: a 401 means it is up and enforcing auth.
        for _ in range(15):
            time.sleep(1)
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{proxy_port}/", timeout=3)
            except urllib.error.HTTPError:
                break
            except Exception:
                continue
            break
        tunnel_target = proxy_port
    else:
        print("WARNING: AUTH_PASSWORD is empty - the public link will be UNPROTECTED.")
        tunnel_target = port

    # --- Expose it through a cloudflared quick tunnel (no signup) ---
    tunnel_log = "/content/cloudflared.log"
    subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{tunnel_target}"],
        stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT,
    )
    public_url = None
    for _ in range(30):
        time.sleep(2)
        with open(tunnel_log) as log_file:
            match = re.search(r"https://[-\w]+\.trycloudflare\.com", log_file.read())
        if match:
            public_url = match.group(0)
            break
    print("=" * 60)
    print(f"Open InvokeAI at: {public_url or 'tunnel not ready - see ' + tunnel_log}")
    if AUTH_PASSWORD:
        print(f"Login: {AUTH_USER} / {AUTH_PASSWORD}")
    print(f"Server log: {server_log}")
    print("=" * 60)

## Step 2 — Stop

Stops the server, proxy and tunnel, then flushes and unmounts Google Drive so all state
is saved. Run it whenever you are done. To also free the GPU afterwards:
`Runtime -> Disconnect and delete runtime`.

In [ ]:
# Step 2 - Stop: shut everything down and persist state to Drive.
# Since Start runs in the background, just run this cell whenever you are done.
import subprocess

for name in ("cloudflared", "caddy", "invokeai-web"):
    subprocess.run(["pkill", "-f", name], check=False)
print("Stopped cloudflared, caddy and invokeai-web (if running).")

try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Google Drive flushed and unmounted - state is saved.")
except Exception as exc:
    print(f"Drive unmount skipped: {exc}")

print("Done. To free the GPU too, use Runtime -> Disconnect and delete runtime.")

## Notes

- **Persistence**: everything under `INVOKEAI_ROOT` (default `My Drive/invokeai`) is on
  Google Drive — models, `outputs`, `databases/invokeai.db` and `invokeai.yaml` all
  survive a runtime stop. Next session just run Start again.
- **Drive location**: set `DRIVE_NAME` (`My Drive` or a Shared Drive name) and
  `INVOKEAI_FOLDER` in the Start form to change where data is stored.
- **Password**: set `AUTH_USER` / `AUTH_PASSWORD`. A Caddy reverse proxy then enforces
  HTTP Basic Auth on `:9091` and cloudflared tunnels that port, so the public link prompts
  for credentials. Empty password = unprotected link.
- **protobuf**: Start upgrades protobuf to >=5.26 because Colab ships an older build that
  crashes diffusers (`cannot import name 'runtime_version'`). Without it the server never
  starts and the tunnel returns 502 Bad Gateway.
- **Downloading models**: use the in-app **Model Manager** — paste a HuggingFace / CivitAI
  / direct URL and InvokeAI downloads it into the Drive root itself. For CivitAI, set your
  API key in Model Manager settings first.
- **Drive space**: large models (Flux, SD3.5 Large, Qwen ~40 GB) consume a lot of Drive
  quota — keep enough free space.
- **SQLite on Drive**: fine for a single user. On a rare "database is locked", run Stop,
  wait a few seconds for Drive to sync, then Start again. Don't open the same root from
  two runtimes at once.
- **Tunnel URL** changes every launch (quick tunnels are ephemeral). For a stable URL,
  switch cloudflared for an authenticated ngrok / cloudflare named tunnel.
- **GPU**: A100 (40 GB) gives the best throughput and fits the largest models; L4/T4 work
  for SDXL/SD1.5 but may need offloading for Flux-class models.